# 06 — Risk and PnL Decomposition
研究回撤、Expected Shortfall、年份稳定性、极端交易贡献，以及 delta/gamma/theta/vega/residual。

In [ ]:
from pathlib import Path
import sys
root=Path.cwd().resolve()
while root!=root.parent and not (root/'pyproject.toml').exists(): root=root.parent
sys.path.insert(0,str(root))
import matplotlib.pyplot as plt
from research.spxw_put_skew.analysis import load_study_context,risk_decomposition
dataset,config,collection,panel,readiness,conclusion=load_study_context(root)
mask=panel.skew_rank>=float(config.high_skew_percentile) if len(panel) else None
summary,trades,by_year=risk_decomposition(panel,mask)
print('Ready:',readiness.ready)
display(summary)
display(by_year)

In [ ]:
if len(trades):
 fig,axes=plt.subplots(2,1,figsize=(12,7),sharex=True)
 axes[0].plot(trades.timestamp,trades.equity,label='cumulative PnL')
 axes[1].fill_between(trades.timestamp,trades.drawdown,0,alpha=.4,label='drawdown')
 for ax in axes: ax.legend(); ax.grid(alpha=.2)
 plt.tight_layout()

In [ ]:
if len(trades):
 components=trades[['delta_pnl','gamma_pnl','theta_pnl','vega_pnl','residual_pnl']].sum()
 display(components.to_frame('PnL'))
 components.plot(kind='bar',title='PnL decomposition'); plt.grid(axis='y',alpha=.2); plt.tight_layout()

In [ ]:
if len(trades):
 display(trades.nsmallest(10,'strategy_pnl')[['timestamp','strategy_pnl','exit_reason','put25_atm_skew','forward_max_drawdown']])
 print('Top 5 absolute trades / total absolute PnL:',trades.strategy_pnl.abs().nlargest(5).sum()/trades.strategy_pnl.abs().sum())

Greeks 分解使用局部近似，residual 包含高阶曲面变化、离散成交与近似误差。Residual 过大时必须回到完整情景重估，不能强行归因。